#### 1. Crawl Data from CafeF

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def crawl_cafef(pages=5):
    """
    Crawl tin tài chính từ CafeF — chuyên mục chứng khoán
    """
    articles = []
    base_url = "https://cafef.vn/thi-truong-chung-khoan.chn"

    for page in range(1, pages + 1):
        url = f"{base_url}?page={page}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, 'html.parser')

            # Lấy danh sách bài viết
            items = soup.select('div.tlitem')  # selector CafeF

            for item in items:
                title_tag = item.select_one('h3 a') or item.select_one('h2 a')
                time_tag  = item.select_one('span.time')

                if not title_tag:
                    continue

                articles.append({
                    'title'  : title_tag.get_text(strip=True),
                    'url'    : 'https://cafef.vn' + title_tag.get('href', ''),
                    'date'   : time_tag.get_text(strip=True) if time_tag else '',
                    'source' : 'cafef'
                })

            print(f"Page {page}: {len(items)} articles")
            time.sleep(1.5)  # tránh bị block

        except Exception as e:
            print(f"Error page {page}: {e}")

    return pd.DataFrame(articles)


def crawl_article_content(url):
    """
    Crawl nội dung bài viết từ URL — dùng để lấy full text cho sentiment
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.text, 'html.parser')

        # CafeF content selector
        content_div = (soup.select_one('div.contentdetail') or
                       soup.select_one('div#mainContent') or
                       soup.select_one('article'))

        if content_div:
            paragraphs = content_div.find_all('p')
            return ' '.join(p.get_text(strip=True) for p in paragraphs)
        return ''

    except Exception as e:
        print(f"Error: {e}")
        return ''

In [3]:
TICKER_KEYWORDS = {
    'VNM': ['Vinamilk', 'VNM'],
    'VIC': ['Vingroup', 'VIC'],
    'HPG': ['Hòa Phát', 'HPG'],
    'VHM': ['Vinhomes', 'VHM'],
    'TCB': ['Techcombank', 'TCB'],
}

def filter_by_ticker(df, ticker):
    """Lọc tin liên quan đến một mã cổ phiếu cụ thể"""
    keywords = TICKER_KEYWORDS.get(ticker, [ticker])
    pattern  = '|'.join(keywords)
    mask     = (df['title'].str.contains(pattern, case=False, na=False) |
                df['content'].str.contains(pattern, case=False, na=False))
    return df[mask].copy()

In [4]:
def build_news_dataset(tickers, pages_per_source=10):
    """
    Pipeline đầy đủ: crawl → lọc theo ticker → lưu CSV
    """
    # 1. Crawl headlines
    print("Crawling CafeF...")
    df = crawl_cafef(pages=pages_per_source)

    # 2. Crawl full content (giới hạn để tránh bị block)
    print("Fetching article content...")
    df['content'] = ''
    for i, row in df.iterrows():
        df.at[i, 'content'] = crawl_article_content(row['url'])
        time.sleep(1)
        if i % 20 == 0:
            print(f"  {i}/{len(df)} done")

    # 3. Lưu raw
    df.to_csv('data/vietnam/news_raw.csv', index=False, encoding='utf-8-sig')

    # 4. Lọc theo từng ticker
    for ticker in tickers:
        filtered = filter_by_ticker(df, ticker)
        filtered.to_csv(f'data/vietnam/news_{ticker}.csv',
                        index=False, encoding='utf-8-sig')
        print(f"{ticker}: {len(filtered)} articles")

    return df

# Chạy
tickers = ['VNM', 'VIC', 'HPG', 'VHM', 'TCB']
df_news  = build_news_dataset(tickers, pages_per_source=20)

Crawling CafeF...
Page 1: 20 articles
Page 2: 20 articles
Page 3: 20 articles
Page 4: 20 articles
Page 5: 20 articles
Page 6: 20 articles
Page 7: 20 articles
Page 8: 20 articles
Page 9: 20 articles
Page 10: 20 articles
Page 11: 20 articles
Page 12: 20 articles
Page 13: 20 articles
Page 14: 20 articles
Page 15: 20 articles
Page 16: 20 articles
Page 17: 20 articles
Page 18: 20 articles
Page 19: 20 articles
Page 20: 20 articles
Fetching article content...
  0/400 done
  20/400 done
  40/400 done
  60/400 done
  80/400 done
  100/400 done
  120/400 done
  140/400 done
  160/400 done
  180/400 done
  200/400 done
  220/400 done
  240/400 done
  260/400 done
  280/400 done
  300/400 done
  320/400 done
  340/400 done
  360/400 done
  380/400 done
VNM: 0 articles
VIC: 100 articles
HPG: 20 articles
VHM: 20 articles
TCB: 20 articles


In [6]:
print(df_news)

                                                 title  \
0    Doanh nghiệp nắm “kho báu” cả thế giới săn lùn...   
1    Vingroup của tỷ phú Phạm Nhật Vượng báo lãi qu...   
2    Đối tác của FPT “tái sinh” nhờ AI, giá trị vượ...   
3    FPT bắt tay Intel triển khai mô hình nhà máy t...   
4    Quỹ vàng khổng lồ không ngừng “xả” hàng khi gi...   
..                                                 ...   
395                  Bất ngờ cách ngân hàng 'giữ tiền'   
396  Thế Giới Di Động đem 48.000 tỷ gửi ngân hàng, ...   
397  “Đại gia” Việt Nam vừa đổ thêm tiền vào Indone...   
398  Công ty robot của ông Phạm Nhật Vượng "bắt tay...   
399  Lợi nhuận PV OIL (OIL) bùng nổ đầu năm, nhưng ...   

                                                   url                 date  \
0    https://cafef.vn/doanh-nghiep-nam-kho-bau-ca-t...  2026-04-28T13:35:00   
1    https://cafef.vn/vingroup-cua-ty-phu-pham-nhat...  2026-04-28T13:32:00   
2    https://cafef.vn/doi-tac-cua-fpt-tai-sinh-nho-...  2026-04-28

In [5]:
# Kiểm tra nhanh chất lượng data sau khi crawl
print(df_news.shape)
print(df_news.isnull().sum())       # content bị null do crawl thất bại
print(df_news.duplicated().sum())   # bài bị trùng giữa các page
print(df_news['content'].str.len().describe())  # bài quá ngắn < 100 ký tự

(400, 5)
title      0
url        0
date       0
source     0
content    0
dtype: int64
380
count     400.000000
mean     3635.500000
std      1467.149846
min      1831.000000
25%      2459.000000
50%      3211.000000
75%      5225.250000
max      6714.000000
Name: content, dtype: float64


In [7]:
# Kiểm tra xem duplicate theo cột nào
print("Duplicate by title:", df_news.duplicated(subset=['title']).sum())
print("Duplicate by url:",   df_news.duplicated(subset=['url']).sum())
print("Duplicate by all:",   df_news.duplicated().sum())

# Xem thử 5 cặp bị trùng
dupes = df_news[df_news.duplicated(subset=['title'], keep=False)]
print(dupes[['title', 'date']].head(10))

Duplicate by title: 380
Duplicate by url: 380
Duplicate by all: 380
                                               title                 date
0  Doanh nghiệp nắm “kho báu” cả thế giới săn lùn...  2026-04-28T13:35:00
1  Vingroup của tỷ phú Phạm Nhật Vượng báo lãi qu...  2026-04-28T13:32:00
2  Đối tác của FPT “tái sinh” nhờ AI, giá trị vượ...  2026-04-28T11:30:00
3  FPT bắt tay Intel triển khai mô hình nhà máy t...  2026-04-28T10:35:00
4  Quỹ vàng khổng lồ không ngừng “xả” hàng khi gi...  2026-04-28T10:23:00
5   Bán lẻ xăng dầu muốn cộng đủ chi phí vào giá bán  2026-04-28T07:22:00
6  Lãnh đạo Hòa Phát, FPT, Masan, HAGL... nói gì ...  2026-04-28T07:21:00
7  Chuyên gia: Đừng nhìn chỉ số để mua, đây là 4 ...  2026-04-28T00:06:00
8  FPT cùng 2 mã ngân hàng có thể được các ETF nộ...  2026-04-28T00:02:00
9  Chứng khoán chỉ giao dịch 2 phiên trong tuần n...  2026-04-27T22:10:00


In [8]:
# keep=False sẽ đánh dấu TẤT CẢ rows liên quan, kể cả bản gốc
dupes = df_news[df_news.duplicated(subset=['title'], keep=False)]

In [9]:
# keep='first' → chỉ đánh dấu bản SAO, giữ lại bản gốc
print("Actual duplicates to remove:", df_news.duplicated(subset=['title'], keep='first').sum())

# Xem thử bài nào thực sự bị lặp
real_dupes = df_news[df_news.duplicated(subset=['title'], keep='first')]
print(real_dupes[['title', 'date']].head())

Actual duplicates to remove: 380
                                                title                 date
20  Doanh nghiệp nắm “kho báu” cả thế giới săn lùn...  2026-04-28T13:35:00
21  Vingroup của tỷ phú Phạm Nhật Vượng báo lãi qu...  2026-04-28T13:32:00
22  Đối tác của FPT “tái sinh” nhờ AI, giá trị vượ...  2026-04-28T11:30:00
23  FPT bắt tay Intel triển khai mô hình nhà máy t...  2026-04-28T10:35:00
24  Quỹ vàng khổng lồ không ngừng “xả” hàng khi gi...  2026-04-28T10:23:00


Comment: This crawling way can only get 20 articles.
Can try with Selenium to get Infinite Scroll. 

In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/147.0.0.0 Safari/537.36',
    'Referer': 'https://cafef.vn/thi-truong-chung-khoan.chn',
    'X-Requested-With': 'XMLHttpRequest',
}

# Test endpoint trước
url = "https://cafef.vn/ajax/list-box-right/18831.chn"
resp = requests.get(url, headers=HEADERS, timeout=10)

print(f"Status : {resp.status_code}")
print(f"Content-Type : {resp.headers.get('Content-Type', '')}")
print(f"Length : {len(resp.text)} chars")
print(f"Preview:\n{resp.text[:500]}")

Status : 200
Content-Type : text/html; charset=utf-8
Length : 5361 chars
Preview:
<style>
        .img-resize {
            display: block;
            position: relative;
            height: -moz-max-content;
            height: max-content;
        }

        .img-resize:before {
            padding-bottom: 62.5%;
            /* 16:10 */
            content: "";
            display: block;
        }

        .img-resize img {
            position: absolute;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            bottom: 0;
   


In [24]:
import feedparser
import requests
import pandas as pd
import time
import os

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/147.0.0.0 Safari/537.36',
}

TICKERS = {
    'VIC' : ['Vingroup', 'VIC cổ phiếu', 'Vingroup chứng khoán'],
    'HPG' : ['Hòa Phát', 'HPG cổ phiếu', 'Hoa Phat steel'],
    'VHM' : ['Vinhomes', 'VHM cổ phiếu'],
    'TCB' : ['Techcombank', 'TCB cổ phiếu'],
    'VNM' : ['Vinamilk', 'VNM cổ phiếu'],
    'FPT' : ['FPT cổ phiếu', 'FPT chứng khoán', 'FPT corporation'],
    'VCB' : ['Vietcombank', 'VCB cổ phiếu'],
    'MSN' : ['Masan', 'MSN cổ phiếu'],
    'MWG' : ['Thế Giới Di Động', 'MWG cổ phiếu'],
    'VND' : ['VNDirect', 'VND chứng khoán'],
}

# ── Step 1: Crawl từ Google News RSS ──────────────────────────────────────
def crawl_google_news(tickers_dict, max_per_query=100):
    all_articles = []

    for ticker, keywords in tickers_dict.items():
        print(f"[{ticker}] Fetching...")
        seen_urls = set()

        for keyword in keywords:
            query = f"{keyword} chứng khoán việt nam"
            url   = (f"https://news.google.com/rss/search"
                     f"?q={requests.utils.quote(query)}"
                     f"&hl=vi&gl=VN&ceid=VN:vi")

            try:
                feed = feedparser.parse(url)
                for entry in feed.entries[:max_per_query]:
                    art_url = entry.get('link', '')
                    if art_url in seen_urls or not art_url:
                        continue
                    seen_urls.add(art_url)

                    all_articles.append({
                        'ticker'  : ticker,
                        'title'   : entry.get('title', '').strip(),
                        'url'     : art_url,
                        'date'    : entry.get('published', ''),
                        'source'  : entry.get('source', {}).get('title', ''),
                        'summary' : entry.get('summary', '').strip(),
                    })
            except Exception as e:
                print(f"  Error [{keyword}]: {e}")

            time.sleep(0.8)

        print(f"  → {len(seen_urls)} articles")
        time.sleep(1)

    return all_articles


# ── Step 2: Fetch content từ URL gốc ──────────────────────────────────────
from bs4 import BeautifulSoup

def fetch_content(google_url):
    """Follow Google News redirect → fetch content từ trang gốc"""
    try:
        # Follow redirect để lấy URL thật
        resp     = requests.get(google_url, headers=HEADERS,
                                timeout=10, allow_redirects=True)
        real_url = resp.url

        # Nếu vẫn là google.com thì skip
        if 'google.com' in real_url:
            return '', ''

        soup = BeautifulSoup(resp.text, 'html.parser')

        # Thử nhiều selector
        content = ''
        for selector in ['article', 'div.contentdetail', 'div.detail-content',
                         'div.article-content', 'div.post-content',
                         'div#mainContent', 'div#content', 'main']:
            div = soup.select_one(selector)
            if div:
                text = ' '.join(p.get_text(strip=True)
                                for p in div.find_all('p')
                                if len(p.get_text(strip=True)) > 20)
                if len(text) > 150:
                    content = text
                    break

        # Fallback: lấy tất cả <p>
        if not content:
            all_p   = soup.find_all('p')
            content = ' '.join(p.get_text(strip=True) for p in all_p
                               if len(p.get_text(strip=True)) > 30)

        return real_url, content[:3000]  # giới hạn 3000 ký tự

    except Exception as e:
        return '', ''


# ── Step 3: Pipeline hoàn chỉnh ───────────────────────────────────────────
def build_dataset(tickers_dict, output_dir='data/vietnam',
                  max_per_query=100, fetch_content_flag=True):

    os.makedirs(output_dir, exist_ok=True)

    # 3.1 Crawl headlines
    print("=" * 50)
    print("STEP 1: Crawling Google News RSS")
    print("=" * 50)
    articles = crawl_google_news(tickers_dict, max_per_query)

    df = pd.DataFrame(articles)

    # Parse date
    df['date'] = pd.to_datetime(df['date'], utc=True, errors='coerce')
    df['date'] = df['date'].dt.tz_convert('Asia/Ho_Chi_Minh').dt.tz_localize(None)
    df = df.dropna(subset=['date'])
    df = df.drop_duplicates(subset=['url']).reset_index(drop=True)
    df = df.sort_values('date', ascending=False).reset_index(drop=True)

    print(f"\nTotal headlines: {len(df)}")
    print(df.groupby('ticker').size().to_string())

    # 3.2 Fetch content
    if fetch_content_flag:
        print("\n" + "=" * 50)
        print("STEP 2: Fetching article content")
        print("=" * 50)

        real_urls, contents = [], []
        for i in range(len(df)):
            real_url, content = fetch_content(df.loc[i, 'url'])
            real_urls.append(real_url)
            contents.append(content)

            if i % 20 == 0:
                success = sum(1 for c in contents if len(c) > 100)
                pct     = success / (i + 1) * 100
                print(f"  {i:>4}/{len(df)} | success: {success} ({pct:.0f}%)"
                      f" | {df.loc[i, 'ticker']} — {df.loc[i, 'title'][:40]}")
            time.sleep(1.2)

        df['real_url'] = real_urls
        df['content']  = contents
    else:
        # Dùng summary từ RSS làm content thay thế
        df['real_url'] = df['url']
        df['content']  = df['summary']

    # 3.3 Filter bài có content
    df['text_for_sentiment'] = df.apply(
        lambda r: f"{r['title']}. {r['content']}"
                  if len(r.get('content', '')) > 50
                  else r['title'],
        axis=1
    )

    # 3.4 Save toàn bộ
    out_all = os.path.join(output_dir, 'news_all.csv')
    df.to_csv(out_all, index=False, encoding='utf-8-sig')
    print(f"\nSaved: {out_all} ({len(df)} articles)")

    # 3.5 Save từng ticker
    for ticker in tickers_dict:
        sub = df[df['ticker'] == ticker].reset_index(drop=True)
        if len(sub) == 0:
            continue
        out_path = os.path.join(output_dir, f'news_{ticker}.csv')
        sub.to_csv(out_path, index=False, encoding='utf-8-sig')
        print(f"  {ticker}: {len(sub):>4} articles → {out_path}")

    # 3.6 Summary
    print("\n" + "=" * 50)
    print("SUMMARY")
    print("=" * 50)
    print(f"Total articles     : {len(df)}")
    print(f"Date range         : {df['date'].min()} → {df['date'].max()}")
    has_content = (df['content'].str.len() > 100).sum()
    print(f"Articles w/content : {has_content} ({has_content/len(df)*100:.0f}%)")
    print(f"Articles title-only: {len(df) - has_content}")
    print(f"\nPer ticker:")
    print(df.groupby('ticker').size().to_string())

    return df


# ── Chạy ──────────────────────────────────────────────────────────────────
df_news = build_dataset(
    tickers_dict      = TICKERS,
    output_dir        = 'data/vietnam',
    max_per_query     = 100,
    fetch_content_flag = True,   # False nếu chỉ muốn title+summary, nhanh hơn
)

STEP 1: Crawling Google News RSS
[VIC] Fetching...
  → 187 articles
[HPG] Fetching...
  → 247 articles
[VHM] Fetching...
  → 186 articles
[TCB] Fetching...
  → 170 articles
[VNM] Fetching...
  → 154 articles
[FPT] Fetching...
  → 196 articles
[VCB] Fetching...
  → 148 articles
[MSN] Fetching...
  → 179 articles
[MWG] Fetching...
  → 192 articles
[VND] Fetching...
  → 191 articles

Total headlines: 1770
ticker
FPT    185
HPG    246
MSN    172
MWG    177
TCB    168
VCB    141
VHM    162
VIC    187
VND    182
VNM    150

STEP 2: Fetching article content
     0/1770 | success: 0 (0%) | VIC — VN-Index tăng mạnh nhờ nhóm cổ phiếu Vin
    20/1770 | success: 0 (0%) | VIC — Pyn Elite Fund: Nhà đầu tư 'ngó lơ' kết 
    40/1770 | success: 0 (0%) | HPG — Thị phần dồn về nhóm dẫn đầu: Cuộc chơi 
    60/1770 | success: 0 (0%) | MWG — Giá RAM, Chip dự báo sẽ còn tăng mạnh, T
    80/1770 | success: 0 (0%) | VIC — Cổ phiếu ngân hàng nổi sóng, VIC và VCB 
   100/1770 | success: 0 (0%) | MWG — Một số cổ 

In [25]:
print("\n--- Số lượng bài viết theo từng mã ---")
print(df_news['ticker'].value_counts())


--- Số lượng bài viết theo từng mã ---
ticker
HPG    246
VIC    187
FPT    185
VND    182
MWG    177
MSN    172
TCB    168
VHM    162
VNM    150
VCB    141
Name: count, dtype: int64


In [28]:
print("\n--- Mẫu văn bản dùng cho Sentiment (3 bài đầu) ---")
for i, text in enumerate(df_news['text_for_sentiment'].head(3)):
    print(f"Bài {i+1}: {text[:200]}...") # In 200 ký tự đầu để kiểm tra
    print("-" * 20)


--- Mẫu văn bản dùng cho Sentiment (3 bài đầu) ---
Bài 1: VN-Index tăng mạnh nhờ nhóm cổ phiếu Vingroup nổi sóng lớn - thuonghieucongluan.com.vn...
--------------------
Bài 2: Vingroup trở thành tập đoàn lớn thứ 4 Đông Nam Á - Znews...
--------------------
Bài 3: Thị trường chứng khoán tăng hơn 22 điểm - Vietnam.vn...
--------------------
